In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
df = pd.read_csv("C://Users//lokes//OneDrive//Desktop//Neural_Network//winequalityN.csv")

In [3]:
df.head()

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,white,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,white,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,white,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [4]:
import tensorflow
import keras

In [5]:
from keras.models import Model
from keras.layers import Dense, Input

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [7]:
df['type'].unique()

array(['white', 'red'], dtype=object)

In [8]:
df['type'] = df['type'].apply(lambda x: 0 if(x=='white') else 1)

In [9]:
num_cols = df.select_dtypes(include=np.number).columns.to_list()
print(len(num_cols))

13


In [10]:
len(df.columns)

13

In [11]:
df.isna().sum().sort_values(ascending=False)

fixed acidity           10
pH                       9
volatile acidity         8
sulphates                4
citric acid              3
chlorides                2
residual sugar           2
type                     0
free sulfur dioxide      0
density                  0
total sulfur dioxide     0
alcohol                  0
quality                  0
dtype: int64

In [12]:
df.dropna(inplace=True)

In [13]:
df.isna().sum()

type                    0
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [14]:
df1 = df.copy()

In [15]:
def get_labels(df):
    wine_type = df.pop('type')
    wine_type = np.array(wine_type)
    quality = df.pop('quality')
    quality = np.array(quality)

    return (wine_type, quality)

In [16]:
train, test = train_test_split(df, test_size=0.2, random_state=42)

In [17]:
train, val = train_test_split(train, test_size=0.2, random_state=42)

In [18]:
print(len(train), len(val), len(test))

4136 1034 1293


In [19]:
train_y = get_labels(train)
val_y = get_labels(val)
test_y = get_labels(test)

In [20]:
sc = StandardScaler()

In [21]:
train_x = sc.fit_transform(train)

In [22]:
val_x = sc.fit_transform(val)

In [23]:
test_x = sc.fit_transform(test)

In [24]:
len(train.columns)

11

In [25]:
def double_prediction_model():
    inp = Input(shape=(11,))

    x = Dense(units=32, activation='relu')(inp)
    x = Dense(units = 32, activation='relu')(x)

    wine_type_layer = Dense(units=1, activation='sigmoid', name='wine_type_layer')(x)

    x2_quality = Dense(units=64, activation='relu', name='x2_quality')(x)
    wine_quality = Dense(units=1, name='wine_quality')(x2_quality)

    multi_model = Model(inputs=inp, outputs=[wine_type_layer, wine_quality])
    return multi_model

In [26]:
model = double_prediction_model()

In [27]:
optimizer = keras.optimizers.RMSprop(learning_rate=0.001)

In [28]:
model.compile(
    loss = {
        'wine_type_layer': 'binary_crossentropy',
        'wine_quality': 'mse'
    }, 
    metrics = {
        'wine_type_layer': 'accuracy',
        'wine_quality': keras.metrics.RootMeanSquaredError()
    }
)

In [29]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 11)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        384 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      1,056 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ x2_quality (Dense)  │ (None, 64)        │      2,112 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wine_type_layer     │ (None, 1)         │         33 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wine_quality        │ (None, 1)         │         65 │ x2_quality[0][0]  │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,650 (14.26 KB)

 Trainable params: 3,650 (14.26 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
import pydot

In [31]:
from tensorflow.python.keras.utils.vis_utils import plot_model

In [32]:
keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


In [33]:
history = model.fit(train_x, train_y, epochs=40, validation_data=(val_x, val_y))

Epoch 1/40
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 6.8420 - wine_quality_loss: 6.1796 - wine_quality_root_mean_squared_error: 2.4913 - wine_type_layer_accuracy: 0.5638 - wine_type_layer_loss: 0.6350 - val_loss: 1.7204 - val_wine_quality_loss: 1.3045 - val_wine_quality_root_mean_squared_error: 1.1471 - val_wine_type_layer_accuracy: 0.7824 - val_wine_type_layer_loss: 0.4027
Epoch 2/40
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.3920 - wine_quality_loss: 1.1178 - wine_quality_root_mean_squared_error: 1.0594 - wine_type_layer_accuracy: 0.9212 - wine_type_layer_loss: 0.2694 - val_loss: 1.0509 - val_wine_quality_loss: 0.8741 - val_wine_quality_root_mean_squared_error: 0.9376 - val_wine_type_layer_accuracy: 0.9729 - val_wine_type_layer_loss: 0.1703
Epoch 3/40
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.9192 - wine_quality_loss: 0.7931 - wine_quality_root_mean_squared_error: 0.8912 - wine_type_layer_accuracy: 0.9860 - wine_type_layer_loss: 0.1249 - val_loss: 0.7741 - val

In [37]:
scores =model.evaluate(x=val_x, y=val_y)

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.5156 - wine_quality_loss: 0.4745 - wine_quality_root_mean_squared_error: 0.6939 - wine_type_layer_accuracy: 0.9942 - wine_type_layer_loss: 0.0334 
